# Week 2: Baseline SCL Classification & Random Forest

This notebook implements:
1. SCL (Scene Classification Layer) baseline extraction
2. Random Forest pixel-level cloud classification
3. Comparison of RF vs SCL baseline metrics

In [ ]:
# Setup: Import preprocessing module
import sys
sys.path.insert(0, '../src')
from preprocessing import CloudSEN12Preprocessor

In [ ]:
# Imports
import numpy as np
import pandas as pd
import json
import rasterio
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_fscore_support, jaccard_score
import tacoreader.v1 as tacoreader

In [ ]:
# Load split info and class info
with open('../outputs/split_info.json') as f:
    split_info = json.load(f)

with open('../outputs/class_info.json') as f:
    class_info = json.load(f)

train_indices = split_info['train_indices']
val_indices = split_info['val_indices']
test_indices = split_info['test_indices']

print(f'Train samples: {len(train_indices)}')
print(f'Val samples: {len(val_indices)}')
print(f'Test samples: {len(test_indices)}')
print(f'\nClass labels: {class_info}')

## 1. Initialize

In [ ]:
# Initialize preprocessor and load dataset
preprocessor = CloudSEN12Preprocessor()
dataset = tacoreader.load("tacofoundation:cloudsen12-l2a")

print(f"✓ Dataset loaded: {len(dataset)} total samples")
print(f"✓ Preprocessor ready - will extract bands {preprocessor.selected_bands}")

## 2. Load Training Data

In [ ]:
print(f"Loading {len(train_indices[:100])} training samples...\n")

X_train_list = []
y_train_list = []

for idx in tqdm(train_indices[:100], desc="Training samples"):
    try:
        # Read sample paths using tacoreader
        sample = dataset.read(idx)
        image_path = sample.read(0)
        target_path = sample.read(1)
        
        # Load image (shape: bands, H, W)
        with rasterio.open(image_path) as src:
            image = src.read()
        
        # Load mask
        with rasterio.open(target_path) as src:
            mask = src.read(1)
        
        if image is None or mask is None:
            continue
        
        # Extract selected bands (RGB+NIR) from (bands, H, W)
        image_selected = preprocessor.extract_bands(image)
        
        # Normalize
        image_norm = preprocessor.normalize_image(image_selected)
        
        # Transpose to (H, W, C) for flattening
        image_hw_c = np.transpose(image_norm, (1, 2, 0))
        
        # Flatten to pixels: (H*W, 4)
        H, W, C = image_hw_c.shape
        pixels = image_hw_c.reshape(-1, C)
        mask_pixels = mask.reshape(-1)
        
        X_train_list.append(pixels)
        y_train_list.append(mask_pixels)
    except Exception as e:
        print(f"Error {idx}: {e}")
        continue

X_train = np.vstack(X_train_list)
y_train = np.concatenate(y_train_list)

print(f"\n✓ Training data: {X_train.shape}")
print(f"✓ Training labels: {y_train.shape}")
print(f"\nClass distribution:\n{pd.Series(y_train).value_counts().sort_index()}")

## 3. Load Validation Data

In [ ]:
print(f"Loading {len(val_indices[:50])} validation samples...\n")

X_val_list = []
y_val_list = []
scl_val_list = []
image_shapes = []
pixel_boundaries = []
pixel_offset = 0

for idx in tqdm(val_indices[:50], desc="Validation samples"):
    try:
        # Read sample paths
        sample = dataset.read(idx)
        image_path = sample.read(0)
        target_path = sample.read(1)
        
        # Load image (shape: bands, H, W)
        with rasterio.open(image_path) as src:
            image = src.read()
        
        # Load mask
        with rasterio.open(target_path) as src:
            mask = src.read(1)
        
        if image is None or mask is None:
            continue
        
        # Extract selected bands
        image_selected = preprocessor.extract_bands(image)
        
        # Normalize
        image_norm = preprocessor.normalize_image(image_selected)
        
        # Transpose to (H, W, C)
        image_hw_c = np.transpose(image_norm, (1, 2, 0))
        H, W, C = image_hw_c.shape
        
        # Store shape for visualization
        image_shapes.append((H, W))
        n_pixels = H * W
        pixel_boundaries.append((pixel_offset, pixel_offset + n_pixels))
        pixel_offset += n_pixels
        
        # Extract SCL baseline (band 11 in original format)
        scl_band = image[11]
        
        # Map SCL to 4 cloud classes
        scl_mapped = np.zeros_like(scl_band)
        scl_mapped[np.isin(scl_band, [2, 4, 5, 6])] = 0  # Clear
        scl_mapped[np.isin(scl_band, [10])] = 1  # Thin
        scl_mapped[np.isin(scl_band, [8, 9])] = 2  # Thick
        scl_mapped[np.isin(scl_band, [3])] = 3  # Shadow
        
        # Flatten
        pixels = image_hw_c.reshape(-1, C)
        mask_pixels = mask.reshape(-1)
        scl_pixels = scl_mapped.reshape(-1)
        
        X_val_list.append(pixels)
        y_val_list.append(mask_pixels)
        scl_val_list.append(scl_pixels)
    except Exception as e:
        print(f"Error {idx}: {e}")
        continue

X_val = np.vstack(X_val_list)
y_val = np.concatenate(y_val_list)
y_scl = np.concatenate(scl_val_list)

print(f"\n✓ Validation data: {X_val.shape}")
print(f"✓ Validation labels: {y_val.shape}")
print(f"✓ Images loaded: {len(image_shapes)}")

## 4. Spectral Features

In [ ]:
def add_spectral_indices(pixels):
    """
    Add spectral indices to RGB+NIR pixel features.
    Input: (N_pixels, 4) - [R, G, B, NIR]
    Output: (N_pixels, 7) - [R, G, B, NIR, NDVI, Brightness, Greenness]
    """
    R, G, B, NIR = pixels[:, 0], pixels[:, 1], pixels[:, 2], pixels[:, 3]
    
    ndvi = (NIR - R) / (NIR + R + 1e-7)
    brightness = np.mean([R, G, B, NIR], axis=0)
    greenness = G / (R + G + B + 1e-7)
    
    return np.column_stack([pixels, ndvi, brightness, greenness])

X_train_feat = add_spectral_indices(X_train)
X_val_feat = add_spectral_indices(X_val)

print(f"Train features: {X_train_feat.shape}")
print(f"Val features: {X_val_feat.shape}")

## 5. Train Random Forest

In [ ]:
print("Training Random Forest...\n")

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=10,
    n_jobs=-1,
    verbose=1,
    random_state=42
)

rf.fit(X_train_feat, y_train)
print("\n✓ Random Forest trained!")

## 6. Feature Importance

In [ ]:
features = ['R', 'G', 'B', 'NIR', 'NDVI', 'Brightness', 'Greenness']
imp = rf.feature_importances_

plt.figure(figsize=(10, 5))
plt.bar(features, imp, color='steelblue')
plt.ylabel('Importance')
plt.title('RF Feature Importance')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../outputs/figures/03_rf_feature_importance.png', dpi=150)
print("✓ Saved: 03_rf_feature_importance.png")
plt.show()

## 7. Predictions

In [ ]:
y_rf = rf.predict(X_val_feat)
y_scl_baseline = y_scl

print(f"RF predictions: {y_rf.shape}")
print(f"SCL predictions: {y_scl_baseline.shape}")

## 8. Metrics

In [ ]:
def metrics(y_true, y_pred, name):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average='macro')
    iou = jaccard_score(y_true, y_pred, average='macro')
    return {'name': name, 'p': p, 'r': r, 'f1': f, 'iou': iou}

rf_met = metrics(y_val, y_rf, 'RF')
scl_met = metrics(y_val, y_scl_baseline, 'SCL')

print("\nRandom Forest:")
print(f"  P={rf_met['p']:.4f}, R={rf_met['r']:.4f}, F1={rf_met['f1']:.4f}, IoU={rf_met['iou']:.4f}")

print("\nSCL Baseline:")
print(f"  P={scl_met['p']:.4f}, R={scl_met['r']:.4f}, F1={scl_met['f1']:.4f}, IoU={scl_met['iou']:.4f}")

# Save
results = {'rf': rf_met, 'scl': scl_met}
with open('../outputs/week2_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("\n✓ Results saved to week2_results.json")

## 9. Visualization

In [ ]:
# Use first image for comparison
s, e = pixel_boundaries[0]
H, W = image_shapes[0]

y_val_img = y_val[s:e].reshape(H, W)
y_rf_img = y_rf[s:e].reshape(H, W)
y_scl_img = y_scl_baseline[s:e].reshape(H, W)

fig, ax = plt.subplots(1, 3, figsize=(15, 5))

im0 = ax[0].imshow(y_val_img, cmap='tab10', vmin=0, vmax=3)
ax[0].set_title('Ground Truth')
ax[0].axis('off')

im1 = ax[1].imshow(y_rf_img, cmap='tab10', vmin=0, vmax=3)
ax[1].set_title('RF Predictions')
ax[1].axis('off')

im2 = ax[2].imshow(y_scl_img, cmap='tab10', vmin=0, vmax=3)
ax[2].set_title('SCL Baseline')
ax[2].axis('off')

cbar = plt.colorbar(im2, ax=ax, orient='horizontal', pad=0.1, shrink=0.8)
cbar.set_label('Class (0=Clear, 1=Thin, 2=Thick, 3=Shadow)')

plt.tight_layout()
plt.savefig('../outputs/figures/04_model_predictions.png', dpi=150, bbox_inches='tight')
print("✓ Saved: 04_model_predictions.png")
plt.show()

# Week 2: Baseline SCL Classification & Random Forest

This notebook implements:
1. SCL (Scene Classification Layer) baseline extraction
2. Random Forest pixel-level cloud classification
3. Comparison of RF vs SCL baseline metrics

In [1]:
# Setup: Import preprocessing module
import sys
sys.path.insert(0, '../src')
from preprocessing import CloudSEN12Preprocessor

In [2]:
# Imports
import numpy as np
import pandas as pd
import json
import rasterio
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_fscore_support, jaccard_score

In [3]:
# Load split info and class info
with open('../outputs/split_info.json') as f:
    split_info = json.load(f)

with open('../outputs/class_info.json') as f:
    class_info = json.load(f)

train_indices = split_info['train_indices']
val_indices = split_info['val_indices']
test_indices = split_info['test_indices']

print(f'Train samples: {len(train_indices)}')
print(f'Val samples: {len(val_indices)}')
print(f'Test samples: {len(test_indices)}')
print(f'\nClass labels: {class_info}')

Train samples: 3014
Val samples: 1004
Test samples: 1006

Class labels: {'num_classes': 4, 'classes': {'0': 'Clear sky', '1': 'Thin cloud', '2': 'Thick cloud', '3': 'Cloud shadow'}, 'distribution': {'0': {'count': 7755982, 'percentage': 59.173446655273445}, '1': {'count': 2735826, 'percentage': 20.872695922851562}, '2': {'count': 1319091, 'percentage': 10.063865661621094}, '3': {'count': 1296301, 'percentage': 9.889991760253906}}}


## 1. Load Training Data & Extract Features

In [4]:
import tacoreader.v1 as tacoreader

# Initialize preprocessor
preprocessor = CloudSEN12Preprocessor()

# Load dataset using tacoreader v1
dataset = tacoreader.load("tacofoundation:cloudsen12-l2a")

print(f"Dataset loaded: {len(dataset)} total samples")
print(f"Training on {len(train_indices[:100])} samples...\n")

# Prepare training data - build pixel-level feature matrix
X_train_list = []
y_train_list = []

for idx in tqdm(train_indices[:100], desc="Loading training samples"):
    try:
        # Read sample paths using tacoreader
        sample = dataset.read(idx)
        image_path = sample.read(0)
        target_path = sample.read(1)
        
        # Load image using rasterio
        with rasterio.open(image_path) as src:
            image = src.read()
        
        # Transpose from (bands, height, width) to (height, width, bands)
        image = np.transpose(image, (1, 2, 0))
        
        # Load cloud mask using rasterio
        with rasterio.open(target_path) as src:
            mask = src.read(1)  # Read first band
        
        if image is None or mask is None:
            continue
        
        # Normalize image
        image_norm = preprocessor.normalize_image(image)
        
        # Extract selected bands (RGB+NIR: indices 3,2,1,7)
        image_bands = preprocessor.extract_bands(image_norm, [3, 2, 1, 7])
        
        # Flatten spatial dimensions to get pixels: (H*W, 4)
        H, W, C = image_bands.shape
        image_pixels = image_bands.reshape(-1, C)
        mask_pixels = mask.reshape(-1)
        
        X_train_list.append(image_pixels)
        y_train_list.append(mask_pixels)
    except Exception as e:
        print(f"Error loading sample {idx}: {e}")
        continue

# Concatenate all training data
X_train = np.vstack(X_train_list)
y_train = np.concatenate(y_train_list)

print(f"\n✓ Training data shape: {X_train.shape}")
print(f"✓ Training labels shape: {y_train.shape}")
print(f"✓ Class distribution:\n{pd.Series(y_train).value_counts().sort_index()}")

Dataset loaded: 50247 total samples
Training on 100 samples...



Loading training samples:   1%|          | 1/100 [00:07<12:51,  7.79s/it]

Error loading sample 0: CloudSEN12Preprocessor.extract_bands() takes 2 positional arguments but 3 were given


Loading training samples:   2%|▏         | 2/100 [00:13<10:34,  6.48s/it]

Error loading sample 1: CloudSEN12Preprocessor.extract_bands() takes 2 positional arguments but 3 were given


...

Error loading sample 6: CloudSEN12Preprocessor.extract_bands() takes 2 positional arguments but 3 were given


Loading training samples:   8%|▊         | 8/100 [00:43<08:00,  5.22s/it]

Error loading sample 7: CloudSEN12Preprocessor.extract_bands() takes 2 positional arguments but 3 were given


Loading training samples:   8%|▊         | 8/100 [00:48<09:12,  6.00s/it]


KeyboardInterrupt: 

## 2. Load Validation Data & Extract SCL Baseline

In [ ]:
# Load validation data and extract SCL baseline
# We'll also store image shapes for later visualization
print(f"Loading validation data...\n")

X_val_list = []
y_val_list = []
scl_val_list = []
image_shapes = []  # Track (H, W) for each image
pixel_boundaries = []  # Track pixel indices for each image

pixel_offset = 0

for idx in tqdm(val_indices[:50], desc="Loading validation samples"):
    try:
        # Read sample paths using tacoreader
        sample = dataset.read(idx)
        image_path = sample.read(0)
        target_path = sample.read(1)
        
        # Load image using rasterio
        with rasterio.open(image_path) as src:
            image = src.read()
        
        # Transpose from (bands, height, width) to (height, width, bands)
        image = np.transpose(image, (1, 2, 0))
        
        # Load cloud mask using rasterio
        with rasterio.open(target_path) as src:
            mask = src.read(1)  # Read first band
        
        if image is None or mask is None:
            continue
        
        # Normalize image
        image_norm = preprocessor.normalize_image(image)
        
        # Extract selected bands (RGB+NIR)
        image_bands = preprocessor.extract_bands(image_norm, [3, 2, 1, 7])
        H, W, C = image_bands.shape
        
        # Store shape info
        image_shapes.append((H, W))
        n_pixels = H * W
        pixel_boundaries.append((pixel_offset, pixel_offset + n_pixels))
        pixel_offset += n_pixels
        
        # Extract SCL baseline (band 11 = Scene Classification Layer)
        scl_band = image[:, :, 11]  # Band 11 is SCL
        
        # Map SCL values to cloud classes
        scl_mapped = np.zeros_like(scl_band)
        scl_mapped[np.isin(scl_band, [2, 4, 5, 6])] = 0  # Clear sky
        scl_mapped[np.isin(scl_band, [10])] = 1  # Thin cloud
        scl_mapped[np.isin(scl_band, [8, 9])] = 2  # Thick cloud
        scl_mapped[np.isin(scl_band, [3])] = 3  # Cloud shadow
        
        # Flatten
        image_pixels = image_bands.reshape(-1, C)
        mask_pixels = mask.reshape(-1)
        scl_pixels = scl_mapped.reshape(-1)
        
        X_val_list.append(image_pixels)
        y_val_list.append(mask_pixels)
        scl_val_list.append(scl_pixels)
    except Exception as e:
        print(f"Error loading validation sample {idx}: {e}")
        continue

# Concatenate validation data
X_val = np.vstack(X_val_list)
y_val = np.concatenate(y_val_list)
y_scl = np.concatenate(scl_val_list)

print(f"\n✓ Validation data shape: {X_val.shape}")
print(f"✓ Validation labels shape: {y_val.shape}")
print(f"✓ SCL baseline shape: {y_scl.shape}")
print(f"✓ Loaded {len(image_shapes)} images")
print(f"✓ Image shapes (first 5): {image_shapes[:5]}")

## 3. Extract Spectral Features

In [ ]:
def add_spectral_indices(image_bands):
    """
    Compute spectral indices from RGB+NIR bands.
    Input: image_bands with shape (N_pixels, 4) - [R, G, B, NIR]
    Output: enhanced features with shape (N_pixels, 7) - [R, G, B, NIR, NDVI, Brightness, Greenness]
    """
    R = image_bands[:, 0]
    G = image_bands[:, 1]
    B = image_bands[:, 2]
    NIR = image_bands[:, 3]
    
    # NDVI (Normalized Difference Vegetation Index)
    ndvi = (NIR - R) / (NIR + R + 1e-7)
    
    # Brightness (average of all bands)
    brightness = np.mean([R, G, B, NIR], axis=0)
    
    # Greenness (normalized green component)
    greenness = G / (R + G + B + 1e-7)
    
    # Stack all features
    features = np.column_stack([image_bands, ndvi, brightness, greenness])
    return features

# Add spectral indices to training and validation data
X_train_features = add_spectral_indices(X_train)
X_val_features = add_spectral_indices(X_val)

print(f"Training features shape: {X_train_features.shape}")
print(f"Validation features shape: {X_val_features.shape}")

## 4. Train Random Forest Classifier

In [ ]:
# Train Random Forest Classifier
print("Training Random Forest...")

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=10,
    n_jobs=-1,
    verbose=1,
    random_state=42
)

rf.fit(X_train_features, y_train)
print("\n✓ Random Forest trained!")

## 5. Feature Importance

In [ ]:
# Plot feature importance
feature_names = ['Red', 'Green', 'Blue', 'NIR', 'NDVI', 'Brightness', 'Greenness']
importances = rf.feature_importances_

plt.figure(figsize=(10, 6))
plt.bar(feature_names, importances, color='steelblue')
plt.xlabel('Feature')
plt.ylabel('Importance')
plt.title('Random Forest Feature Importance')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../outputs/figures/03_rf_feature_importance.png', dpi=150)
print("✓ Feature importance saved!")
plt.show()

## 6. Make Predictions on Validation Set

In [ ]:
# Make predictions
y_rf_pred = rf.predict(X_val_features)
y_scl_pred = y_scl  # SCL baseline predictions

print(f"RF predictions shape: {y_rf_pred.shape}")
print(f"SCL predictions shape: {y_scl_pred.shape}")

## 7. Compute Metrics

In [ ]:
def compute_metrics(y_true, y_pred, model_name):
    """
    Compute precision, recall, F1, and IoU metrics.
    """
    # Macro-averaged metrics
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='macro')
    iou = jaccard_score(y_true, y_pred, average='macro')
    
    # Per-class metrics
    precision_per_class, recall_per_class, f1_per_class, _ = precision_recall_fscore_support(
        y_true, y_pred, average=None, labels=[0, 1, 2, 3]
    )
    iou_per_class = []
    for class_idx in [0, 1, 2, 3]:
        mask = y_true == class_idx
        if mask.sum() > 0:
            iou_per_class.append(jaccard_score(y_true[mask], y_pred[mask], average='binary'))
        else:
            iou_per_class.append(0.0)
    
    return {
        'model': model_name,
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1),
        'iou': float(iou),
        'precision_per_class': precision_per_class.tolist(),
        'recall_per_class': recall_per_class.tolist(),
        'f1_per_class': f1_per_class.tolist(),
        'iou_per_class': iou_per_class
    }

# Compute metrics for both models
rf_metrics = compute_metrics(y_val, y_rf_pred, 'RandomForest')
scl_metrics = compute_metrics(y_val, y_scl_pred, 'SCL_Baseline')

print("\n" + "="*60)
print("RANDOM FOREST RESULTS")
print("="*60)
print(f"Precision (macro): {rf_metrics['precision']:.4f}")
print(f"Recall (macro):    {rf_metrics['recall']:.4f}")
print(f"F1-score (macro):  {rf_metrics['f1']:.4f}")
print(f"IoU (macro):       {rf_metrics['iou']:.4f}")
print(f"\nPer-class metrics:")
for i, label in enumerate(['Clear Sky', 'Thin Cloud', 'Thick Cloud', 'Cloud Shadow']):
    print(f"  {label}: P={rf_metrics['precision_per_class'][i]:.4f}, R={rf_metrics['recall_per_class'][i]:.4f}, F1={rf_metrics['f1_per_class'][i]:.4f}, IoU={rf_metrics['iou_per_class'][i]:.4f}")

print("\n" + "="*60)
print("SCL BASELINE RESULTS")
print("="*60)
print(f"Precision (macro): {scl_metrics['precision']:.4f}")
print(f"Recall (macro):    {scl_metrics['recall']:.4f}")
print(f"F1-score (macro):  {scl_metrics['f1']:.4f}")
print(f"IoU (macro):       {scl_metrics['iou']:.4f}")
print(f"\nPer-class metrics:")
for i, label in enumerate(['Clear Sky', 'Thin Cloud', 'Thick Cloud', 'Cloud Shadow']):
    print(f"  {label}: P={scl_metrics['precision_per_class'][i]:.4f}, R={scl_metrics['recall_per_class'][i]:.4f}, F1={scl_metrics['f1_per_class'][i]:.4f}, IoU={scl_metrics['iou_per_class'][i]:.4f}")

## 8. Results Comparison Table

In [ ]:
# Create comparison table
comparison_data = {
    'Metric': ['Precision (macro)', 'Recall (macro)', 'F1-score (macro)', 'IoU (macro)'],
    'Random Forest': [
        f"{rf_metrics['precision']:.4f}",
        f"{rf_metrics['recall']:.4f}",
        f"{rf_metrics['f1']:.4f}",
        f"{rf_metrics['iou']:.4f}"
    ],
    'SCL Baseline': [
        f"{scl_metrics['precision']:.4f}",
        f"{scl_metrics['recall']:.4f}",
        f"{scl_metrics['f1']:.4f}",
        f"{scl_metrics['iou']:.4f}"
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print("\n" + "="*60)
print("COMPARISON: Random Forest vs SCL Baseline")
print("="*60)
print(comparison_df.to_string(index=False))

# Save results
results = {
    'random_forest': rf_metrics,
    'scl_baseline': scl_metrics
}

with open('../outputs/week2_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("\n✓ Results saved to week2_results.json")

## 9. Visualization: RF vs SCL Predictions

In [ ]:
# Create visualization comparing RF and SCL predictions
# Use the first image for visualization
start_idx, end_idx = pixel_boundaries[0]
H, W = image_shapes[0]

# Reshape predictions back to image space
y_val_reshaped = y_val[start_idx:end_idx].reshape(H, W)
y_rf_reshaped = y_rf_pred[start_idx:end_idx].reshape(H, W)
y_scl_reshaped = y_scl_pred[start_idx:end_idx].reshape(H, W)

# Create 3-panel figure
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Ground truth
im0 = axes[0].imshow(y_val_reshaped, cmap='tab10', vmin=0, vmax=3)
axes[0].set_title('Ground Truth (Label Mask)')
axes[0].axis('off')

# RF predictions
im1 = axes[1].imshow(y_rf_reshaped, cmap='tab10', vmin=0, vmax=3)
axes[1].set_title('Random Forest Predictions')
axes[1].axis('off')

# SCL baseline
im2 = axes[2].imshow(y_scl_reshaped, cmap='tab10', vmin=0, vmax=3)
axes[2].set_title('SCL Baseline Predictions')
axes[2].axis('off')

# Add colorbar
cbar = plt.colorbar(im2, ax=axes, orientation='horizontal', pad=0.05, shrink=0.8)
cbar.set_label('Class (0=Clear, 1=Thin, 2=Thick, 3=Shadow)')

plt.tight_layout()
plt.savefig('../outputs/figures/04_model_predictions.png', dpi=150, bbox_inches='tight')
print("✓ Prediction visualization saved!")
plt.show()

# Week 2: Baseline and Classical ML Model

This notebook implements:
1. **SCL Baseline** - Sentinel-2 Scene Classification Layer (band 11)
2. **Random Forest** - Pixel-level cloud classification
3. **Comparison** - Metrics and performance analysis

**Goal**: Establish a working ML model and baseline for future comparisons.

## 1. Environment Setup & Imports

In [ ]:
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, jaccard_score
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.insert(0, '../src')
from preprocessing import CloudSEN12Preprocessor
from config import Config

print("✓ All imports successful")

✓ All imports successful


## 2. Load Configuration and Split Info

In [ ]:
# Load split information
with open('../outputs/split_info.json', 'r') as f:
    split_info = json.load(f)

train_indices = np.array(split_info['train_indices'])
val_indices = np.array(split_info['val_indices'])
test_indices = np.array(split_info['test_indices'])

print(f"✓ Train set: {len(train_indices)} samples")
print(f"✓ Validation set: {len(val_indices)} samples")
print(f"✓ Test set: {len(test_indices)} samples")

# Load class info
with open('../outputs/class_info.json', 'r') as f:
    class_info = json.load(f)

class_labels = class_info['classes']
print(f"\n✓ Classes: {class_labels}")

✓ Train set: 3014 samples
✓ Validation set: 1004 samples
✓ Test set: 1006 samples

✓ Classes: {'0': 'Clear sky', '1': 'Thin cloud', '2': 'Thick cloud', '3': 'Cloud shadow'}


## 3. Load and Prepare Training Data

In [ ]:
import tacoreader.v1 as tacoreader
from tqdm import tqdm

# Initialize preprocessor
preprocessor = CloudSEN12Preprocessor()

# Load dataset using tacoreader v1
dataset = tacoreader.load("tacofoundation:cloudsen12-l2a")

print(f"Dataset loaded: {len(dataset)} total samples")
print(f"Training on {len(train_indices[:100])} samples...\n")

# Prepare training data - build pixel-level feature matrix
X_train_list = []
y_train_list = []

for idx in tqdm(train_indices[:100], desc="Loading training samples"):
    try:
        # Read sample paths using tacoreader
        sample = dataset.read(idx)
        image_path = sample.read(0)
        target_path = sample.read(1)
        
        # Load image using rasterio
        with rasterio.open(image_path) as src:
            image = src.read()
        
        # Transpose from (bands, height, width) to (height, width, bands)
        image = np.transpose(image, (1, 2, 0))
        
        # Load cloud mask using rasterio
        with rasterio.open(target_path) as src:
            mask = src.read(1)  # Read first band
        
        if image is None or mask is None:
            continue
        
        # Normalize image
        image_norm = preprocessor.normalize_image(image)
        
        # Extract selected bands (RGB+NIR: indices 3,2,1,7)
        image_bands = preprocessor.extract_bands(image_norm, [3, 2, 1, 7])
        
        # Flatten spatial dimensions to get pixels: (H*W, 4)
        H, W, C = image_bands.shape
        image_pixels = image_bands.reshape(-1, C)
        mask_pixels = mask.reshape(-1)
        
        X_train_list.append(image_pixels)
        y_train_list.append(mask_pixels)
    except Exception as e:
        print(f"Error loading sample {idx}: {e}")
        continue

# Concatenate all training data
X_train = np.vstack(X_train_list)
y_train = np.concatenate(y_train_list)

print(f"\n✓ Training data shape: {X_train.shape}")
print(f"✓ Training labels shape: {y_train.shape}")
print(f"✓ Class distribution:\n{pd.Series(y_train).value_counts().sort_index()}")

TacoFormatError: [93m⚠️  [1mLegacy TACO v1 format detected:[0m [96mtacofoundation:cloudsen12-l2a[0m

[91mtacoreader 2.0+ does not support v1 formats (.taco, .tortilla, tacofoundation:).[0m

[1mTo read legacy files:[0m
  1. Install tacoreader v0.x:
       [92mpip install 'tacoreader<1.0'[0m

  2. Migrate your dataset to [96m.tacozip[0m format using [96mtacotoolbox 2.0[0m

  3. [1m[93mIMPORTANT:[0m Use the legacy import path:

         [92mimport tacoreader.v1 as tacoreader[0m

     instead of:
         [91mimport tacoreader[0m

[1mWe recommend migrating to the new PIT-based format for better performance.[0m

## 4. Extract SCL Baseline on Validation Set

In [ ]:
# Load validation data and extract SCL baseline
# We'll also store image shapes for later visualization
print(f"Loading validation data...\n")

X_val_list = []
y_val_list = []
scl_val_list = []
image_shapes = []  # Track (H, W) for each image
pixel_boundaries = []  # Track pixel indices for each image

pixel_offset = 0

for idx in tqdm(val_indices[:50], desc="Loading validation samples"):
    try:
        # Read sample paths using tacoreader
        sample = dataset.read(idx)
        image_path = sample.read(0)
        target_path = sample.read(1)
        
        # Load image using rasterio
        with rasterio.open(image_path) as src:
            image = src.read()
        
        # Transpose from (bands, height, width) to (height, width, bands)
        image = np.transpose(image, (1, 2, 0))
        
        # Load cloud mask using rasterio
        with rasterio.open(target_path) as src:
            mask = src.read(1)  # Read first band
        
        if image is None or mask is None:
            continue
        
        # Normalize image
        image_norm = preprocessor.normalize_image(image)
        
        # Extract selected bands (RGB+NIR)
        image_bands = preprocessor.extract_bands(image_norm, [3, 2, 1, 7])
        H, W, C = image_bands.shape
        
        # Store shape info
        image_shapes.append((H, W))
        n_pixels = H * W
        pixel_boundaries.append((pixel_offset, pixel_offset + n_pixels))
        pixel_offset += n_pixels
        
        # Extract SCL baseline (band 11 = Scene Classification Layer)
        scl_band = image[:, :, 11]  # Band 11 is SCL
        
        # Map SCL values to cloud classes
        # SCL: 0=No Data, 1=Saturated/Defective, 2=Dark/Shaded, 3=Cloud Shadow, 4=Vegetation, 
        #      5=Not Vegetated, 6=Water, 7=Unclassified, 8=Cloud Medium, 9=Cloud High, 10=Thin Cirrus, 11=Snow
        # Cloud detection mapping:
        #   0 (Clear sky): SCL in [2,4,5,6] = Dark, Vegetation, Not Veg, Water
        #   1 (Thin cloud): SCL in [10] = Thin Cirrus
        #   2 (Thick cloud): SCL in [8,9] = Medium/High clouds
        #   3 (Cloud shadow): SCL in [3] = Shadow
        scl_mapped = np.zeros_like(scl_band)
        scl_mapped[np.isin(scl_band, [2, 4, 5, 6])] = 0  # Clear sky
        scl_mapped[np.isin(scl_band, [10])] = 1  # Thin cloud
        scl_mapped[np.isin(scl_band, [8, 9])] = 2  # Thick cloud
        scl_mapped[np.isin(scl_band, [3])] = 3  # Cloud shadow
        
        # Flatten
        image_pixels = image_bands.reshape(-1, C)
        mask_pixels = mask.reshape(-1)
        scl_pixels = scl_mapped.reshape(-1)
        
        X_val_list.append(image_pixels)
        y_val_list.append(mask_pixels)
        scl_val_list.append(scl_pixels)
    except Exception as e:
        print(f"Error loading validation sample {idx}: {e}")
        continue

# Concatenate validation data
X_val = np.vstack(X_val_list)
y_val = np.concatenate(y_val_list)
y_scl = np.concatenate(scl_val_list)

print(f"\n✓ Validation data shape: {X_val.shape}")
print(f"✓ Validation labels shape: {y_val.shape}")
print(f"✓ SCL baseline shape: {y_scl.shape}")
print(f"✓ Loaded {len(image_shapes)} images")
print(f"✓ Image shapes (first 5): {image_shapes[:5]}")

## 5. Extract Spectral Features

In [ ]:
def add_spectral_indices(image_pixels):
    """
    Add spectral indices to feature matrix.
    Input: (N_pixels, 4) where columns are [R, G, B, NIR]
    Output: (N_pixels, 7) with added features
    """
    R = image_pixels[:, 0]
    G = image_pixels[:, 1]
    B = image_pixels[:, 2]
    NIR = image_pixels[:, 3]
    
    # Avoid division by zero
    eps = 1e-8
    
    # NDVI: (NIR - R) / (NIR + R)
    NDVI = (NIR - R) / (NIR + R + eps)
    
    # NDBI: (SWIR - NIR) / (SWIR + NIR) - approximate with (B - NIR) / (B + NIR)
    # Since we don't have SWIR, use a simple brightness index
    Brightness = (R + G + B + NIR) / 4
    
    # Greenness: G / (R + G + B)
    Greenness = G / (R + G + B + eps)
    
    # Stack all features
    features = np.column_stack([
        image_pixels,  # R, G, B, NIR (4 features)
        NDVI,
        Brightness,
        Greenness
    ])
    
    return features

# Add spectral indices to training and validation data
X_train_features = add_spectral_indices(X_train)
X_val_features = add_spectral_indices(X_val)

print(f"✓ Training features shape: {X_train_features.shape}")
print(f"✓ Validation features shape: {X_val_features.shape}")
print(f"\n✓ Feature names: R, G, B, NIR, NDVI, Brightness, Greenness")

## 6. Train Random Forest Classifier

In [ ]:
print("Training Random Forest classifier...")
print(f"Training samples: {X_train_features.shape[0]}")
print(f"Features: {X_train_features.shape[1]}\n")

# Train Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=10,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

rf_model.fit(X_train_features, y_train)

print("\n✓ Random Forest training complete!")
print(f"✓ Model trained on {X_train_features.shape[0]} pixels")

## 7. Feature Importance Analysis

In [ ]:
# Feature importance
feature_names = ['Red', 'Green', 'Blue', 'NIR', 'NDVI', 'Brightness', 'Greenness']
importances = rf_model.feature_importances_

# Sort by importance
indices = np.argsort(importances)[::-1]

print("Feature Importance (Random Forest):")
print("-" * 40)
for i in range(len(feature_names)):
    idx = indices[i]
    print(f"{i+1}. {feature_names[idx]:<15} {importances[idx]:.4f}")

# Plot feature importance
plt.figure(figsize=(10, 5))
plt.bar(range(len(importances)), importances[indices])
plt.xticks(range(len(importances)), [feature_names[i] for i in indices], rotation=45)
plt.ylabel('Importance')
plt.title('Random Forest Feature Importance for Cloud Detection')
plt.tight_layout()
plt.savefig('../outputs/figures/03_rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Feature importance plot saved")

## 8. Predictions on Validation Set

In [ ]:
print("Generating predictions on validation set...\n")

# RF predictions
y_rf_pred = rf_model.predict(X_val_features)
y_rf_pred_proba = rf_model.predict_proba(X_val_features)

# SCL predictions (already computed)
y_scl_pred = y_scl

print(f"✓ RF predictions shape: {y_rf_pred.shape}")
print(f"✓ SCL predictions shape: {y_scl_pred.shape}")
print(f"✓ Ground truth shape: {y_val.shape}")

## 9. Compute Evaluation Metrics

In [ ]:
def compute_metrics(y_true, y_pred, model_name):
    """
    Compute Precision, Recall, F1-score, IoU for all classes
    """
    metrics_dict = {}
    
    # Overall metrics
    precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    iou_macro = jaccard_score(y_true, y_pred, average='macro', zero_division=0)
    
    metrics_dict['model'] = model_name
    metrics_dict['precision_macro'] = precision_macro
    metrics_dict['recall_macro'] = recall_macro
    metrics_dict['f1_macro'] = f1_macro
    metrics_dict['iou_macro'] = iou_macro
    
    # Per-class metrics
    for cls in range(4):
        mask = y_true == cls
        if mask.sum() == 0:
            continue
        
        y_true_binary = (y_true == cls).astype(int)
        y_pred_binary = (y_pred == cls).astype(int)
        
        precision = precision_score(y_true_binary, y_pred_binary, zero_division=0)
        recall = recall_score(y_true_binary, y_pred_binary, zero_division=0)
        f1 = f1_score(y_true_binary, y_pred_binary, zero_division=0)
        iou = jaccard_score(y_true_binary, y_pred_binary, zero_division=0)
        
        metrics_dict[f'class_{cls}_precision'] = precision
        metrics_dict[f'class_{cls}_recall'] = recall
        metrics_dict[f'class_{cls}_f1'] = f1
        metrics_dict[f'class_{cls}_iou'] = iou
    
    return metrics_dict

# Compute metrics for both models
rf_metrics = compute_metrics(y_val, y_rf_pred, 'Random Forest')
scl_metrics = compute_metrics(y_val, y_scl_pred, 'SCL Baseline')

print("✓ Metrics computed")

## 10. Results Comparison Table

In [ ]:
# Create comparison table
class_names = ['Clear Sky', 'Thin Cloud', 'Thick Cloud', 'Cloud Shadow']

print("\n" + "="*80)
print("OVERALL METRICS COMPARISON")
print("="*80)
print(f"{'Metric':<20} {'Random Forest':<20} {'SCL Baseline':<20} {'Improvement':<20}")
print("-"*80)

metrics_to_compare = ['precision_macro', 'recall_macro', 'f1_macro', 'iou_macro']
for metric in metrics_to_compare:
    rf_val = rf_metrics[metric]
    scl_val = scl_metrics[metric]
    improvement = ((rf_val - scl_val) / scl_val * 100) if scl_val != 0 else 0
    
    metric_name = metric.replace('_', ' ').title()
    print(f"{metric_name:<20} {rf_val:<20.4f} {scl_val:<20.4f} {improvement:+.2f}%")

print("\n" + "="*80)
print("PER-CLASS METRICS")
print("="*80)

for cls in range(4):
    print(f"\n{class_names[cls].upper()}")
    print("-"*80)
    print(f"{'Metric':<20} {'Random Forest':<20} {'SCL Baseline':<20}")
    print("-"*80)
    
    for metric in ['precision', 'recall', 'f1', 'iou']:
        rf_key = f'class_{cls}_{metric}'
        scl_key = f'class_{cls}_{metric}'
        rf_val = rf_metrics.get(rf_key, 0)
        scl_val = scl_metrics.get(scl_key, 0)
        
        print(f"{metric.upper():<20} {rf_val:<20.4f} {scl_val:<20.4f}")

## 11. Visualization: Model Predictions vs Ground Truth

In [ ]:
# Visualize predictions using stored image shapes
# Pick the first image as example

if len(image_shapes) > 0:
    H, W = image_shapes[0]
    start_idx, end_idx = pixel_boundaries[0]
    
    # Extract data for this sample
    y_val_sample = y_val[start_idx:end_idx].reshape(H, W)
    y_rf_sample = y_rf_pred[start_idx:end_idx].reshape(H, W)
    y_scl_sample = y_scl_pred[start_idx:end_idx].reshape(H, W)
    
    # Plot comparison
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    im0 = axes[0].imshow(y_val_sample, cmap='tab10', vmin=0, vmax=3)
    axes[0].set_title('Ground Truth')
    axes[0].axis('off')
    
    im1 = axes[1].imshow(y_rf_sample, cmap='tab10', vmin=0, vmax=3)
    axes[1].set_title('Random Forest Prediction')
    axes[1].axis('off')
    
    im2 = axes[2].imshow(y_scl_sample, cmap='tab10', vmin=0, vmax=3)
    axes[2].set_title('SCL Baseline')
    axes[2].axis('off')
    
    # Add colorbar
    cbar = fig.colorbar(im0, ax=axes, orientation='horizontal', pad=0.05, shrink=0.8)
    cbar.set_label('Cloud Class')
    
    plt.tight_layout()
    plt.savefig('../outputs/figures/04_model_predictions.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Prediction visualization saved")
    print(f"✓ Image shape: {H} x {W}")
else:
    print("No validation images loaded")

## 12. Save Results and Model

In [ ]:
import pickle

# Save Random Forest model
with open('../outputs/random_forest_model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

print("✓ Random Forest model saved")

# Save metrics
results = {
    'random_forest_metrics': rf_metrics,
    'scl_baseline_metrics': scl_metrics,
    'training_info': {
        'n_train_samples': len(train_indices[:100]),
        'n_val_samples': len(val_indices[:50]),
        'n_train_pixels': X_train_features.shape[0],
        'n_val_pixels': X_val_features.shape[0],
        'n_features': X_train_features.shape[1],
        'class_names': class_names,
        'image_shapes': image_shapes
    }
}

with open('../outputs/week2_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("✓ Results saved to outputs/week2_results.json")

## 13. Week 2 Summary

In [ ]:
print("\n" + "="*80)
print("WEEK 2 SUMMARY: BASELINE AND RANDOM FOREST")
print("="*80)

print("\n✓ COMPLETED TASKS:")
print("  1. Loaded CloudSEN12 training data (100 samples)")
print("  2. Extracted spectral features (R, G, B, NIR, NDVI, Brightness, Greenness)")
print("  3. Trained Random Forest classifier (100 estimators, max_depth=20)")
print(f"  4. Evaluated on validation set ({len(val_indices[:50])} samples)")
print("  5. Extracted SCL baseline (Sentinel-2 Scene Classification Layer)")
print("  6. Computed metrics: Precision, Recall, F1-score, IoU")
print("  7. Compared Random Forest vs SCL baseline")

print("\n✓ KEY RESULTS:")
print(f"  Random Forest F1-Score: {rf_metrics['f1_macro']:.4f}")
print(f"  SCL Baseline F1-Score:  {scl_metrics['f1_macro']:.4f}")
improvement = ((rf_metrics['f1_macro'] - scl_metrics['f1_macro']) / scl_metrics['f1_macro'] * 100)
print(f"  Improvement: {improvement:+.2f}%")

print("\n✓ ARTIFACTS GENERATED:")
print("  - outputs/random_forest_model.pkl (trained model)")
print("  - outputs/week2_results.json (metrics and results)")
print("  - outputs/figures/03_rf_feature_importance.png")
print("  - outputs/figures/04_model_predictions.png")

print("\n✓ READY FOR WEEK 3: U-Net Deep Learning Model")
print("="*80)

# Week 2: Baseline and Classical ML Model

This notebook implements:
1. **SCL Baseline** - Sentinel-2 Scene Classification Layer (band 11)
2. **Random Forest** - Pixel-level cloud classification
3. **Comparison** - Metrics and performance analysis

**Goal**: Establish a working ML model and baseline for future comparisons.

## 1. Environment Setup & Imports

In [ ]:
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, jaccard_score
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.insert(0, '../src')
from preprocessing import CloudSEN12Preprocessor
from config import Config

print("✓ All imports successful")

## 2. Load Configuration and Split Info

In [ ]:
# Load split information
with open('../outputs/split_info.json', 'r') as f:
    split_info = json.load(f)

train_indices = np.array(split_info['train_indices'])
val_indices = np.array(split_info['val_indices'])
test_indices = np.array(split_info['test_indices'])

print(f"✓ Train set: {len(train_indices)} samples")
print(f"✓ Validation set: {len(val_indices)} samples")
print(f"✓ Test set: {len(test_indices)} samples")

# Load class info
with open('../outputs/class_info.json', 'r') as f:
    class_info = json.load(f)

class_labels = class_info['classes']
print(f"\n✓ Classes: {class_labels}")

## 3. Load and Prepare Training Data

In [ ]:
import tacoreader
from tqdm import tqdm

# Initialize preprocessor
preprocessor = CloudSEN12Preprocessor()

# Load dataset
dataset = tacoreader.load("tacofoundation:cloudsen12-l2a")

print(f"Dataset loaded: {len(dataset)} total samples")
print(f"Training on {len(train_indices)} samples...\n")

# Prepare training data - build pixel-level feature matrix
X_train_list = []
y_train_list = []

for idx in tqdm(train_indices[:100], desc="Loading training samples"):  # Using 100 samples for training
    try:
        # Load image and mask
        image = preprocessor.load_image(dataset, idx)
        mask = preprocessor.load_mask(dataset, idx)
        
        if image is None or mask is None:
            continue
        
        # Normalize image
        image_norm = preprocessor.normalize_image(image)
        
        # Extract selected bands (RGB+NIR: indices 3,2,1,7)
        image_bands = preprocessor.extract_bands(image_norm, [3, 2, 1, 7])
        
        # Flatten spatial dimensions to get pixels: (H*W, 4)
        H, W, C = image_bands.shape
        image_pixels = image_bands.reshape(-1, C)
        mask_pixels = mask.reshape(-1)
        
        X_train_list.append(image_pixels)
        y_train_list.append(mask_pixels)
    except Exception as e:
        print(f"Error loading sample {idx}: {e}")
        continue

# Concatenate all training data
X_train = np.vstack(X_train_list)
y_train = np.concatenate(y_train_list)

print(f"\n✓ Training data shape: {X_train.shape}")
print(f"✓ Training labels shape: {y_train.shape}")
print(f"✓ Class distribution:\n{pd.Series(y_train).value_counts().sort_index()}")

## 4. Extract SCL Baseline on Validation Set

In [ ]:
# Load validation data and extract SCL baseline
print(f"Loading validation data...\n")

X_val_list = []
y_val_list = []
scl_val_list = []

for idx in tqdm(val_indices[:50], desc="Loading validation samples"):  # Using 50 samples for validation
    try:
        # Load image and mask
        image = preprocessor.load_image(dataset, idx)
        mask = preprocessor.load_mask(dataset, idx)
        
        if image is None or mask is None:
            continue
        
        # Normalize image
        image_norm = preprocessor.normalize_image(image)
        
        # Extract selected bands (RGB+NIR)
        image_bands = preprocessor.extract_bands(image_norm, [3, 2, 1, 7])
        
        # Extract SCL baseline (band 11 = Scene Classification Layer)
        scl_band = image[11]  # Band 11 is SCL
        
        # Map SCL values to cloud classes
        # SCL: 0=No Data, 1=Saturated/Defective, 2=Dark/Shaded, 3=Cloud Shadow, 4=Vegetation, 
        #      5=Not Vegetated, 6=Water, 7=Unclassified, 8=Cloud Medium, 9=Cloud High, 10=Thin Cirrus, 11=Snow
        # Cloud detection mapping:
        #   0 (Clear sky): SCL in [2,4,5,6] = Dark, Vegetation, Not Veg, Water
        #   1 (Thin cloud): SCL in [10] = Thin Cirrus
        #   2 (Thick cloud): SCL in [8,9] = Medium/High clouds
        #   3 (Cloud shadow): SCL in [3] = Shadow
        scl_mapped = np.zeros_like(scl_band)
        scl_mapped[np.isin(scl_band, [2, 4, 5, 6])] = 0  # Clear sky
        scl_mapped[np.isin(scl_band, [10])] = 1  # Thin cloud
        scl_mapped[np.isin(scl_band, [8, 9])] = 2  # Thick cloud
        scl_mapped[np.isin(scl_band, [3])] = 3  # Cloud shadow
        
        # Flatten
        H, W, C = image_bands.shape
        image_pixels = image_bands.reshape(-1, C)
        mask_pixels = mask.reshape(-1)
        scl_pixels = scl_mapped.reshape(-1)
        
        X_val_list.append(image_pixels)
        y_val_list.append(mask_pixels)
        scl_val_list.append(scl_pixels)
    except Exception as e:
        print(f"Error loading validation sample {idx}: {e}")
        continue

# Concatenate validation data
X_val = np.vstack(X_val_list)
y_val = np.concatenate(y_val_list)
y_scl = np.concatenate(scl_val_list)

print(f"\n✓ Validation data shape: {X_val.shape}")
print(f"✓ Validation labels shape: {y_val.shape}")
print(f"✓ SCL baseline shape: {y_scl.shape}")

## 5. Extract Spectral Features

In [ ]:
def add_spectral_indices(image_pixels):
    """
    Add spectral indices to feature matrix.
    Input: (N_pixels, 4) where columns are [R, G, B, NIR]
    Output: (N_pixels, 7) with added features
    """
    R = image_pixels[:, 0]
    G = image_pixels[:, 1]
    B = image_pixels[:, 2]
    NIR = image_pixels[:, 3]
    
    # Avoid division by zero
    eps = 1e-8
    
    # NDVI: (NIR - R) / (NIR + R)
    NDVI = (NIR - R) / (NIR + R + eps)
    
    # NDBI: (SWIR - NIR) / (SWIR + NIR) - approximate with (B - NIR) / (B + NIR)
    # Since we don't have SWIR, use a simple brightness index
    Brightness = (R + G + B + NIR) / 4
    
    # Greenness: G / (R + G + B)
    Greenness = G / (R + G + B + eps)
    
    # Stack all features
    features = np.column_stack([
        image_pixels,  # R, G, B, NIR (4 features)
        NDVI,
        Brightness,
        Greenness
    ])
    
    return features

# Add spectral indices to training and validation data
X_train_features = add_spectral_indices(X_train)
X_val_features = add_spectral_indices(X_val)

print(f"✓ Training features shape: {X_train_features.shape}")
print(f"✓ Validation features shape: {X_val_features.shape}")
print(f"\n✓ Feature names: R, G, B, NIR, NDVI, Brightness, Greenness")

## 6. Train Random Forest Classifier

In [ ]:
print("Training Random Forest classifier...")
print(f"Training samples: {X_train_features.shape[0]}")
print(f"Features: {X_train_features.shape[1]}\n")

# Train Random Forest
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    min_samples_split=10,
    n_jobs=-1,
    random_state=42,
    verbose=1
)

rf_model.fit(X_train_features, y_train)

print("\n✓ Random Forest training complete!")
print(f"✓ Model trained on {X_train_features.shape[0]} pixels")

## 7. Feature Importance Analysis

In [ ]:
# Feature importance
feature_names = ['Red', 'Green', 'Blue', 'NIR', 'NDVI', 'Brightness', 'Greenness']
importances = rf_model.feature_importances_

# Sort by importance
indices = np.argsort(importances)[::-1]

print("Feature Importance (Random Forest):")
print("-" * 40)
for i in range(len(feature_names)):
    idx = indices[i]
    print(f"{i+1}. {feature_names[idx]:<15} {importances[idx]:.4f}")

# Plot feature importance
plt.figure(figsize=(10, 5))
plt.bar(range(len(importances)), importances[indices])
plt.xticks(range(len(importances)), [feature_names[i] for i in indices], rotation=45)
plt.ylabel('Importance')
plt.title('Random Forest Feature Importance for Cloud Detection')
plt.tight_layout()
plt.savefig('../outputs/figures/03_rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Feature importance plot saved")

## 8. Predictions on Validation Set

In [ ]:
print("Generating predictions on validation set...\n")

# RF predictions
y_rf_pred = rf_model.predict(X_val_features)
y_rf_pred_proba = rf_model.predict_proba(X_val_features)

# SCL predictions (already computed)
y_scl_pred = y_scl

print(f"✓ RF predictions shape: {y_rf_pred.shape}")
print(f"✓ SCL predictions shape: {y_scl_pred.shape}")
print(f"✓ Ground truth shape: {y_val.shape}")

## 9. Compute Evaluation Metrics

In [ ]:
def compute_metrics(y_true, y_pred, model_name):
    """
    Compute Precision, Recall, F1-score, IoU for all classes
    """
    metrics_dict = {}
    
    # Overall metrics
    precision_macro = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0)
    iou_macro = jaccard_score(y_true, y_pred, average='macro', zero_division=0)
    
    metrics_dict['model'] = model_name
    metrics_dict['precision_macro'] = precision_macro
    metrics_dict['recall_macro'] = recall_macro
    metrics_dict['f1_macro'] = f1_macro
    metrics_dict['iou_macro'] = iou_macro
    
    # Per-class metrics
    for cls in range(4):
        mask = y_true == cls
        if mask.sum() == 0:
            continue
        
        y_true_binary = (y_true == cls).astype(int)
        y_pred_binary = (y_pred == cls).astype(int)
        
        precision = precision_score(y_true_binary, y_pred_binary, zero_division=0)
        recall = recall_score(y_true_binary, y_pred_binary, zero_division=0)
        f1 = f1_score(y_true_binary, y_pred_binary, zero_division=0)
        iou = jaccard_score(y_true_binary, y_pred_binary, zero_division=0)
        
        metrics_dict[f'class_{cls}_precision'] = precision
        metrics_dict[f'class_{cls}_recall'] = recall
        metrics_dict[f'class_{cls}_f1'] = f1
        metrics_dict[f'class_{cls}_iou'] = iou
    
    return metrics_dict

# Compute metrics for both models
rf_metrics = compute_metrics(y_val, y_rf_pred, 'Random Forest')
scl_metrics = compute_metrics(y_val, y_scl_pred, 'SCL Baseline')

print("✓ Metrics computed")

## 10. Results Comparison Table

In [ ]:
# Create comparison table
class_names = ['Clear Sky', 'Thin Cloud', 'Thick Cloud', 'Cloud Shadow']

print("\n" + "="*80)
print("OVERALL METRICS COMPARISON")
print("="*80)
print(f"{'Metric':<20} {'Random Forest':<20} {'SCL Baseline':<20} {'Improvement':<20}")
print("-"*80)

metrics_to_compare = ['precision_macro', 'recall_macro', 'f1_macro', 'iou_macro']
for metric in metrics_to_compare:
    rf_val = rf_metrics[metric]
    scl_val = scl_metrics[metric]
    improvement = ((rf_val - scl_val) / scl_val * 100) if scl_val != 0 else 0
    
    metric_name = metric.replace('_', ' ').title()
    print(f"{metric_name:<20} {rf_val:<20.4f} {scl_val:<20.4f} {improvement:+.2f}%")

print("\n" + "="*80)
print("PER-CLASS METRICS")
print("="*80)

for cls in range(4):
    print(f"\n{class_names[cls].upper()}")
    print("-"*80)
    print(f"{'Metric':<20} {'Random Forest':<20} {'SCL Baseline':<20}")
    print("-"*80)
    
    for metric in ['precision', 'recall', 'f1', 'iou']:
        rf_key = f'class_{cls}_{metric}'
        scl_key = f'class_{cls}_{metric}'
        rf_val = rf_metrics.get(rf_key, 0)
        scl_val = scl_metrics.get(scl_key, 0)
        
        print(f"{metric.upper():<20} {rf_val:<20.4f} {scl_val:<20.4f}")

## 11. Visualization: Model Predictions vs Ground Truth

In [ ]:
# Reshape predictions back to image format for visualization
# Using a subset of validation images

sample_idx = 0
H, W = 512, 512  # CloudSEN12 tile size

# Find a suitable sample to visualize
pixels_per_sample = H * W
sample_start = sample_idx * pixels_per_sample
sample_end = sample_start + pixels_per_sample

if sample_end <= len(y_val):
    # Extract data for this sample
    y_val_sample = y_val[sample_start:sample_end].reshape(H, W)
    y_rf_sample = y_rf_pred[sample_start:sample_end].reshape(H, W)
    y_scl_sample = y_scl_pred[sample_start:sample_end].reshape(H, W)
    
    # Plot comparison
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    im0 = axes[0].imshow(y_val_sample, cmap='tab10', vmin=0, vmax=3)
    axes[0].set_title('Ground Truth')
    axes[0].axis('off')
    
    im1 = axes[1].imshow(y_rf_sample, cmap='tab10', vmin=0, vmax=3)
    axes[1].set_title('Random Forest Prediction')
    axes[1].axis('off')
    
    im2 = axes[2].imshow(y_scl_sample, cmap='tab10', vmin=0, vmax=3)
    axes[2].set_title('SCL Baseline')
    axes[2].axis('off')
    
    # Add colorbar
    cbar = fig.colorbar(im0, ax=axes, orientation='horizontal', pad=0.05, shrink=0.8)
    cbar.set_label('Cloud Class')
    
    plt.tight_layout()
    plt.savefig('../outputs/figures/04_model_predictions.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("✓ Prediction visualization saved")
else:
    print("Not enough validation data for visualization")

## 12. Save Results and Model

In [ ]:
import pickle

# Save Random Forest model
with open('../outputs/random_forest_model.pkl', 'wb') as f:
    pickle.dump(rf_model, f)

print("✓ Random Forest model saved")

# Save metrics
results = {
    'random_forest_metrics': rf_metrics,
    'scl_baseline_metrics': scl_metrics,
    'training_info': {
        'n_train_samples': len(train_indices[:100]),
        'n_val_samples': len(val_indices[:50]),
        'n_train_pixels': X_train_features.shape[0],
        'n_val_pixels': X_val_features.shape[0],
        'n_features': X_train_features.shape[1],
        'class_names': class_names
    }
}

with open('../outputs/week2_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("✓ Results saved to outputs/week2_results.json")

## 13. Week 2 Summary

In [ ]:
print("\n" + "="*80)
print("WEEK 2 SUMMARY: BASELINE AND RANDOM FOREST")
print("="*80)

print("\n✓ COMPLETED TASKS:")
print("  1. Loaded CloudSEN12 training data (100 samples)")
print("  2. Extracted spectral features (R, G, B, NIR, NDVI, Brightness, Greenness)")
print("  3. Trained Random Forest classifier (100 estimators, max_depth=20)")
print(f"  4. Evaluated on validation set ({len(val_indices[:50])} samples)")
print("  5. Extracted SCL baseline (Sentinel-2 Scene Classification Layer)")
print("  6. Computed metrics: Precision, Recall, F1-score, IoU")
print("  7. Compared Random Forest vs SCL baseline")

print("\n✓ KEY RESULTS:")
print(f"  Random Forest F1-Score: {rf_metrics['f1_macro']:.4f}")
print(f"  SCL Baseline F1-Score:  {scl_metrics['f1_macro']:.4f}")
improvement = ((rf_metrics['f1_macro'] - scl_metrics['f1_macro']) / scl_metrics['f1_macro'] * 100)
print(f"  Improvement: {improvement:+.2f}%")

print("\n✓ ARTIFACTS GENERATED:")
print("  - outputs/random_forest_model.pkl (trained model)")
print("  - outputs/week2_results.json (metrics and results)")
print("  - outputs/figures/03_rf_feature_importance.png")
print("  - outputs/figures/04_model_predictions.png")

print("\n✓ READY FOR WEEK 3: U-Net Deep Learning Model")
print("="*80)